In [4]:
import pandas as pd # import pandas library — the standard Python tool for working with tabular data

In [5]:
# This reads the key-value pairs from the .env file and loads them into the current process's environment variables. 
# It returns True if it successfully found and loaded a .env file, or False if it couldn't find one
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from openai import OpenAI # imports the OpenAI client class from the openai Python package.
openai_client = OpenAI() # creates an instance of that client, which handles authentication and communication with OpenAI's API.

# Generating Ground Truth Data

In [7]:
# We need questions with known relevant documents.
# We generate these with an LLM, asking it to create questions for each exercise:

# Loading the documents

In [8]:
df = pd.read_csv('../data/data.csv')
documents = df.to_dict(orient='records')

In [9]:
# We'll generate questions for the fitness dataset. 
# The full Fitness dataset contains documents that cover different muscle groups, equipment types, and difficulty levels.

# prompt template for question generation

In [10]:
prompt_template = """
You are a user of a fitness assistant app. Based on the exercise record below,
generate 5 questions a user might ask that this record would be the best answer to.
The questions should be complete and not too short — imagine a real user typing them
without seeing the record itself.

Exercise record:
exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}

Provide the output in parsable JSON, as a list of 5 strings, with no other text:

["question1", "question2", ..., "question5"]
""".strip()

# function to call the LLM for one record

In [11]:
documents[0]

{'id': 'push-up-001',
 'exercise_name': 'Push-Up',
 'type_of_activity': 'Strength',
 'type_of_equipment': 'None (bodyweight)',
 'body_part': 'Chest',
 'type': 'Compound',
 'muscle_groups_activated': 'Chest, Triceps, Shoulders, Core',
 'instructions': 'Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.'}

In [12]:
prompt = prompt_template.format(**documents[0])

In [13]:
prompt

'You are a user of a fitness assistant app. Based on the exercise record below,\ngenerate 5 questions a user might ask that this record would be the best answer to.\nThe questions should be complete and not too short — imagine a real user typing them\nwithout seeing the record itself.\n\nExercise record:\nexercise_name: Push-Up\ntype_of_activity: Strength\ntype_of_equipment: None (bodyweight)\nbody_part: Chest\ntype: Compound\nmuscle_groups_activated: Chest, Triceps, Shoulders, Core\ninstructions: Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.\n\nProvide the output in parsable JSON, as a list of 5 strings, with no other text:\n\n["question1", "question2", ..., "question5"]'

In [14]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [15]:
questions = llm(prompt)

In [16]:
questions

'["What are some effective strength exercises I can do without any equipment?", "How can I perform push-ups correctly to ensure I’m getting the most benefit?", "What muscles are engaged when doing push-ups, and which part of the body do they primarily target?", "Can you provide me with step-by-step instructions for doing push-ups to improve my form?", "Is the push-up a good compound exercise, and what are the benefits of including it in my workout routine?"]'

# Generating Ground Truth for All Documents

In [17]:
import json
from tqdm.auto import tqdm

results = {}

for doc in tqdm(documents):
    doc_id = doc['id']
    if doc_id in results:
        continue  # skip if already processed (lets you safely re-run after a failure)
    
    prompt = prompt_template.format(**doc)
    questions_raw = llm(prompt)
    results[doc_id] = questions_raw

  0%|          | 0/50 [00:00<?, ?it/s]

In [18]:
# --- Parse the raw JSON strings into actual lists ---
parsed_results = {}

for doc_id, questions_raw in results.items():
    try:
        parsed_results[doc_id] = json.loads(questions_raw)
    except json.JSONDecodeError:
        print(f"Failed to parse for {doc_id}: {questions_raw[:200]}")


# --- Flatten into (id, question) pairs ---
final_results = []

for doc_id, questions in parsed_results.items():
    for q in questions:
        final_results.append((doc_id, q))

In [19]:
print(len(final_results))       # should be ~250 (50 exercises × 5 questions)
print(final_results[:3])        # should show clean (id, question) tuples

250
[('push-up-001', 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?'), ('push-up-001', "Can you explain the proper technique for performing a push-up to ensure I don't injure myself?"), ('push-up-001', 'What muscle groups are activated when doing push-ups, and how can they benefit my overall fitness?')]


In [20]:
final_results[0]

('push-up-001',
 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?')

In [21]:
df_ground_truth = pd.DataFrame(final_results, columns=['id', 'question'])
df_ground_truth.to_csv('../data/ground-truth-data.csv', index=False)

In [22]:
!head ../data/ground-truth-data.csv

id,question
push-up-001,What are some effective bodyweight exercises I can do to strengthen my chest muscles?
push-up-001,Can you explain the proper technique for performing a push-up to ensure I don't injure myself?
push-up-001,"What muscle groups are activated when doing push-ups, and how can they benefit my overall fitness?"
push-up-001,"Is there any special equipment needed to perform push-ups, or can they be done anywhere?"
push-up-001,What are the benefits of incorporating compound exercises like push-ups into my strength training routine?
bodyweight-squat-002,What are some effective bodyweight exercises I can do to strengthen my legs without any equipment?
bodyweight-squat-002,Can you give me detailed instructions on how to perform a bodyweight squat correctly?
bodyweight-squat-002,What muscle groups are activated when performing bodyweight squats?
bodyweight-squat-002,"Is the bodyweight squat considered a compound exercise, and if so, why?"


In [23]:
df_ground_truth

,id,question
0,push-up-001,What are some effective bodyweight exercises I...
1,push-up-001,Can you explain the proper technique for perfo...
2,push-up-001,What muscle groups are activated when doing pu...
3,push-up-001,Is there any special equipment needed to perfo...
4,push-up-001,What are the benefits of incorporating compoun...
...,...,...
245,donkey-kick-050,What is the proper technique for performing a ...
246,donkey-kick-050,Can you tell me about a strength exercise that...
247,donkey-kick-050,What muscle groups are activated when I perfor...
248,donkey-kick-050,How do I make sure I'm doing the Donkey Kick c...


In [24]:
df_ground_truth.head() # see top 5

,id,question
0,push-up-001,What are some effective bodyweight exercises I...
1,push-up-001,Can you explain the proper technique for perfo...
2,push-up-001,What muscle groups are activated when doing pu...
3,push-up-001,Is there any special equipment needed to perfo...
4,push-up-001,What are the benefits of incorporating compoun...


In [25]:
len(df_ground_truth)

250

In [26]:
ground_truth = df_ground_truth.to_dict(orient='records') #easier to deal with dictionaries

In [27]:
ground_truth [0]

{'id': 'push-up-001',
 'question': 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?'}

In [28]:
q = ground_truth[0]
q

{'id': 'push-up-001',
 'question': 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?'}

In [29]:
doc_id = q['id']
doc_id

'push-up-001'

# Evaluating retrieval quality


In [30]:
from ingest import load_fitness_data, build_index

documents = load_fitness_data()
index = build_index(documents)

In [31]:
import os
print(os.getcwd())

/Users/vincentaina/fitness-assistant-practice/fitness-assistant/notebooks


In [32]:
# Now, to compute Hit Rate and MRR, you need three things:
# A ground-truth dataset — a list of queries, each paired with the id of the document that should be retrieved 
# A search function that returns ranked results for each query
# The metric functions themselves

In [33]:
query = "what exercise is similar to push up?"

In [34]:
boost = {} # no field is treated as more important than another

index.search(
        query=query,
        filter_dict={},  # no filtering. The search runs across everything in the index
        boost_dict=boost,
        num_results=10 # returns the top 10 matches.
    )


[{'id': 'push-up-001',
  'exercise_name': 'Push-Up',
  'type_of_activity': 'Strength',
  'type_of_equipment': 'None (bodyweight)',
  'body_part': 'Chest',
  'type': 'Compound',
  'muscle_groups_activated': 'Chest, Triceps, Shoulders, Core',
  'instructions': 'Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.'},
 {'id': 'pull-up-007',
  'exercise_name': 'Pull-Up',
  'type_of_activity': 'Strength',
  'type_of_equipment': 'Pull-up Bar',
  'body_part': 'Back',
  'type': 'Compound',
  'muscle_groups_activated': 'Lats, Biceps, Rhomboids, Rear Delts, Core',
  'instructions': 'Grip the pull-up bar slightly wider than shoulder-width with palms facing away. Hang with your arms fully extended and core engaged. Pull your chest toward the bar by driving

# Define hit rate and mrr

In [35]:
# hit_rate = (number of queries where correct doc appears in results) / total queries
# MRR (Mean Reciprocal Rank) - When the correct doc does show up, how high up was it ranked?" — this rewards ranking quality, not just presence.

In [36]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

# Search Evaluation

In [37]:
# Define search function

In [38]:
def minsearch_search(query, boost=None):
    if boost is None: # no field is treated as more important than another
        boost = {}
    results = index.search(
        query=query,
        filter_dict={}, # no filtering. The search runs across everything in the index
        boost_dict=boost,
        num_results=10  # returns the top 10 matches.
    )
    return results

In [39]:
results

{'push-up-001': '[\n    "What are some effective bodyweight exercises I can do to strengthen my chest muscles?",\n    "Can you explain the proper technique for performing a push-up to ensure I don\'t injure myself?",\n    "What muscle groups are activated when doing push-ups, and how can they benefit my overall fitness?",\n    "Is there any special equipment needed to perform push-ups, or can they be done anywhere?",\n    "What are the benefits of incorporating compound exercises like push-ups into my strength training routine?"\n]',
 'bodyweight-squat-002': '[\n    "What are some effective bodyweight exercises I can do to strengthen my legs without any equipment?",\n    "Can you give me detailed instructions on how to perform a bodyweight squat correctly?",\n    "What muscle groups are activated when performing bodyweight squats?",\n    "Is the bodyweight squat considered a compound exercise, and if so, why?",\n    "How should I position my feet and body when doing bodyweight squats t

In [40]:
# Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.
# For each question in our ground truth dataset (250 total) we run search. Then we check whether the results include the correct document.
# For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

In [41]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for item in tqdm(ground_truth):
        doc_id = item['id']
        results = search_function(item['question'])
        relevance = [d['id'] == doc_id for d in results] # take each item out of the results list, one at a time, and temporarily call it d while working with it."
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [42]:
result = evaluate(ground_truth, lambda q: minsearch_search(q))
print(result)

  0%|          | 0/250 [00:00<?, ?it/s]

{'hit_rate': 0.996, 'mrr': 0.9479666666666667}


In [43]:
# finding the best boost_dict values guessing by hand and seeing how it influences the hit rate and mrr values

In [44]:
def minsearch_search2(query):
    boost = {"muscle_groups_activated": 2.0, "instructions": 0.5}

    results = index.search(
        query=query,
        filter_dict={},  # no filtering. The search runs across everything in the index
        boost_dict=boost,
        num_results=10 # returns the top 10 matches.
    )

    return results

In [45]:
result = evaluate(ground_truth, lambda q: minsearch_search2(q))
print(result)

  0%|          | 0/250 [00:00<?, ?it/s]

{'hit_rate': 0.996, 'mrr': 0.9492}


In [46]:
# finding the best boost_dict values without guessing by hand

In [47]:
# Given we have 7 boost fields (exercise_name, type_of_activity, type_of_equipment, body_part, type, muscle_groups_activated, instructions)
# grid search is impractical (too many combinations). Random search is the right tool here.

# Split the ground truth and implement random search to find best boost values

In [48]:
# This is a classic ML practice, applied here to search tuning: don't tune and evaluate on the same data.

In [49]:

# Validation set (first 100 questions) — used to search for the best boost parameters.
# Test set (remaining questions) — held back, untouched during tuning. Once you've picked your "best" boost params using validation data, you check performance on the test set once — this tells you how well your tuning generalizes, rather than just overfitting to quirks of the validation set.
# If you tuned and evaluated on the exact same 250 questions, you'd risk picking boost values that happen to work great on those specific questions but don't generalize — similar to overfitting in ML.

gt_val = ground_truth[:100]  
gt_test = ground_truth[100:] 

print(len(gt_val))   # 100
print(len(gt_test))  # 150

100
150


In [50]:
param_ranges = {
    'exercise_name': (0.0, 3.0),
    'type_of_activity': (0.0, 3.0),
    'type_of_equipment': (0.0, 3.0),
    'body_part': (0.0, 3.0),
    'type': (0.0, 3.0),
    'muscle_groups_activated': (0.0, 3.0),
    'instructions': (0.0, 3.0),
}

def objective(boost_params):
    def search_function(q):
        return minsearch_search(q, boost_params)

    results = evaluate(gt_val, search_function)
    return results['mrr']

In [51]:
import random # Bring in Python's built-in random module — needed for generating random numbers (random.randint, random.uniform).

def simple_optimize(param_ranges, objective_function, n_iterations=10): 
# param_ranges — a dict describing, for each parameter, the min/max bounds to search within.\
# objective_function — a function you hand in that takes one set of parameters and returns a score, it returns the MRR)
# n_iterations=10 — how many random guesses to try. =10 is a default value
    best_params = None # will eventually hold whichever parameter combination scored highest. Starts as None because nothing has been tried yet.
    best_score = float('-inf')  # negative infinity. This is a common trick: start your "best score so far" at the lowest possible value, so that literally any real score you compute will beat it
    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items(): # param = the key, (min_val, max_val) = the value, .items() lets you loop over a dictionary and get both the key and value together,
            if isinstance(min_val, int) and isinstance(max_val, int): # whole numbers and decimals need different random function, checks if min_val and max_val are int or not
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [52]:
simple_optimize(param_ranges, objective, n_iterations=20)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

({'exercise_name': 2.768256870486524,
  'type_of_activity': 2.5312179853263763,
  'type_of_equipment': 0.38323261618057813,
  'body_part': 2.6708588161697198,
  'type': 1.3359258110156391,
  'muscle_groups_activated': 1.246509232681984,
  'instructions': 0.8222473754907844},
 0.9686666666666668)

In [53]:
def minsearch_improved(query):
    boost = {
        'exercise_name': 2.53,
        'type_of_activity': 0.69,
        'type_of_equipment': 2.09,
        'body_part': 2.23,
        'type': 2.94,
        'muscle_groups_activated': 0.87,
        'instructions': 1.59
    }

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

evaluate(ground_truth, lambda q: minsearch_improved(q))

  0%|          | 0/250 [00:00<?, ?it/s]

{'hit_rate': 0.996, 'mrr': 0.9544666666666667}

# RAG Evaluation

In [54]:
print(ground_truth[0])

{'id': 'push-up-001', 'question': 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?'}


In [55]:
print(documents[0])

{'id': 'push-up-001', 'exercise_name': 'Push-Up', 'type_of_activity': 'Strength', 'type_of_equipment': 'None (bodyweight)', 'body_part': 'Chest', 'type': 'Compound', 'muscle_groups_activated': 'Chest, Triceps, Shoulders, Core', 'instructions': 'Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.'}


In [78]:
question = 'Is the Lat Pulldown considered a strength training activity, and if so, why?'
answer = rag(question)
print(answer)

Yes, the Lat Pulldown is considered a strength training activity because it targets multiple muscle groups, including the lats, biceps, rhomboids, and rear deltoids, and is performed using a machine. It is categorized as a compound exercise, which involves the use of multiple joints and muscle groups, contributing to overall strength development in the back.


In [80]:
prompt_template = """

You're a fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

Question:
{question}

Context:
{context}
"""

In [81]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [82]:
ground_truth

[{'id': 'push-up-001',
  'question': 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?'},
 {'id': 'push-up-001',
  'question': "Can you explain the proper technique for performing a push-up to ensure I don't injure myself?"},
 {'id': 'push-up-001',
  'question': 'What muscle groups are activated when doing push-ups, and how can they benefit my overall fitness?'},
 {'id': 'push-up-001',
  'question': 'Is there any special equipment needed to perform push-ups, or can they be done anywhere?'},
 {'id': 'push-up-001',
  'question': 'What are the benefits of incorporating compound exercises like push-ups into my strength training routine?'},
 {'id': 'bodyweight-squat-002',
  'question': 'What are some effective bodyweight exercises I can do to strengthen my legs without any equipment?'},
 {'id': 'bodyweight-squat-002',
  'question': 'Can you give me detailed instructions on how to perform a bodyweight squat correctly?'},
 {'id': 'bodyweight-squat-002',
  

In [83]:
ground_truth [0]

{'id': 'push-up-001',
 'question': 'What are some effective bodyweight exercises I can do to strengthen my chest muscles?'}

In [84]:
len(ground_truth)

250

In [85]:
df_sample = df_ground_truth.sample(n=100, random_state=1)

In [86]:
df_sample

,id,question
67,lat-pulldown-014,What type of equipment do I need for a lat pul...
249,donkey-kick-050,Could you provide step-by-step instructions fo...
230,sled-push-047,What is the Sled Push exercise and what type o...
161,incline-bench-press-033,Can you provide detailed instructions for perf...
91,seated-row-019,Which muscles are activated when doing a Seate...
...,...,...
17,lunges-004,What muscle groups are activated when doing Fo...
5,bodyweight-squat-002,What are some effective bodyweight exercises I...
185,smith-machine-squat-038,What is the proper technique for performing a ...
106,burpee-022,Can you tell me about the muscle groups target...


In [87]:
sample = df_sample.to_dict(orient='records')

In [88]:
sample

[{'id': 'lat-pulldown-014',
  'question': 'What type of equipment do I need for a lat pulldown workout?'},
 {'id': 'donkey-kick-050',
  'question': "Could you provide step-by-step instructions for the Donkey Kick to ensure I'm maximizing my workout?"},
 {'id': 'sled-push-047',
  'question': 'What is the Sled Push exercise and what type of activity is it considered?'},
 {'id': 'incline-bench-press-033',
  'question': 'Can you provide detailed instructions for performing an incline dumbbell bench press?'},
 {'id': 'seated-row-019',
  'question': 'Which muscles are activated when doing a Seated Cable Row?'},
 {'id': 'pallof-press-045',
  'question': 'What body parts are activated when I do the Pallof Press on a cable machine?'},
 {'id': 'barbell-squat-012',
  'question': 'Can you explain how to safely execute the Barbell Back Squat for strength training?'},
 {'id': 'sled-push-047',
  'question': 'How can I maintain good form while doing the Sled Push to avoid injury?'},
 {'id': 'hack-squa

In [89]:
# The context is a formatted string with all the search results:

def build_context(results):
    lines = []
    for r in results:
        entry = (
            f"exercise_name: {r['exercise_name']}\n"
            f"type_of_activity: {r['type_of_activity']}\n"
            f"type_of_equipment: {r['type_of_equipment']}\n"
            f"body_part: {r['body_part']}\n"
            f"type: {r['type']}\n"
            f"muscle_groups_activated: {r['muscle_groups_activated']}\n"
            f"instructions: {r['instructions']}"
        )
        lines.append(entry)
    context = "\n\n".join(lines)
    return context

In [90]:
def build_prompt(query, results):
    context = build_context(results)
    prompt = prompt_template.format(question=query, context=context)
    return prompt.strip()

In [91]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [92]:
def llm(prompt, model='gpt-4o-mini'):
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]  
    )
    return response.choices[0].message.content

In [93]:
def rag(query, model='gpt-4o-mini'):
    search_results = minsearch_search(query)
    prompt = build_prompt(query, search_results)
    #print(prompt)
    answer = llm(prompt, model=model)
    return answer

In [94]:
from tqdm.auto import tqdm

In [95]:
evaluations = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question) 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)

    evaluations.append((record, answer_llm, evaluation))

  0%|          | 0/100 [00:00<?, ?it/s]

In [96]:
df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [97]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.93
PARTLY_RELEVANT    0.06
NON_RELEVANT       0.01
Name: proportion, dtype: float64

In [98]:
df_eval.to_csv('../data/rag-eval-gpt-4o-mini.csv', index=False)

In [99]:
df_eval[df_eval.relevance == 'NON_RELEVANT']

,answer,id,question,relevance,explanation
98,The context provided does not include informat...,burpee-022,Can you tell me about the muscle groups target...,NON_RELEVANT,The generated answer does not address the ques...


In [100]:
df_eval[df_eval.relevance == 'PARTLY_RELEVANT']

,answer,id,question,relevance,explanation
31,To perform the Dumbbell Shoulder Press effecti...,shoulder-press-009,What equipment do I need to perform the Dumbbe...,PARTLY_RELEVANT,The generated answer identifies a key piece of...
32,"For a Romanian Deadlift workout, the equipment...",romanian-deadlift-013,Can you tell me about the equipment needed for...,PARTLY_RELEVANT,"The generated answer mentions a barbell, which..."
55,"To do a Russian Twist at home, you don't need ...",russian-twist-024,What type of equipment do I need to do a Russi...,PARTLY_RELEVANT,The generated answer provides information that...
69,The lat pulldown mainly targets the back.,lat-pulldown-014,What specific body part does the lat pulldown ...,PARTLY_RELEVANT,The generated answer identifies the general ar...
82,To properly execute a Barbell Hip Thrust exerc...,hip-thrust-032,What equipment do I need to properly execute a...,PARTLY_RELEVANT,The generated answer correctly identifies that...
85,"To perform the Pallof Press exercise, you need...",pallof-press-045,What type of equipment do I need to perform th...,PARTLY_RELEVANT,The generated answer correctly identifies one ...
